In [66]:
from dysts import flows

In [73]:
lorenz  = flows.Lorenz()
lorenz1 = lorenz.make_trajectory(5000, resample=True, points_per_period=50, standardize=True, solver='RK45', rtol=1e-3)
lorenz2 = lorenz.make_trajectory(5000, resample=True, points_per_period=50, standardize=True, solver='RK45', rtol=1e-10)

c:\Users\jaych\ReservoirGrid\venv\Lib\site-packages\scipy\integrate\_ivp\ivp.py:621: UserWarning:

The following arguments have no effect for a chosen solver: `points_per_period`, `solver`.



In [68]:
from reservoirgrid.helpers import viz

viz.compare_components([lorenz1, lorenz2], labels=['rtol=1e-3', 'rtol=1e-10'])

In [69]:
from reservoirgrid.models import Reservoir
from reservoirgrid.helpers import chaos_utils
from reservoirgrid.helpers import utils

chaos_utils.lyapunov_time(lorenz1[:, 0], lorenz2[:, 0])

445.0

In [70]:
import optuna

#lorenz2 = utils.normalize_data(lorenz2)
X_train, X_val, Y_train, Y_val = utils.split(lorenz2) 

model = Reservoir(input_dim=3, reservoir_dim=1000, output_dim=3)
best_params = model.optimize(
    X_train=X_train,
    Y_train=Y_train,
    X_val=X_val,
    Y_val=Y_val,
    sampler=optuna.samplers.CmaEsSampler(),
    metric_fn=utils.RMSE,
    #metric_fn=chaos_utils.js_divergence,
    direction="minimize",
    n_trials=120,
    batch_size=20,
)
predictions = model.predict(initial_input=Y_train, steps=len(Y_val))

predictions = predictions.cpu().numpy().squeeze(1)
Y_val = Y_val.cpu().numpy()

[I 2026-05-29 16:34:40,187] A new study created in memory with name: no-name-85dc3b7d-c14d-471c-aa2f-66b6e2167c40


[Reservoir] Starting in-class optimization using 'RMSE' (minimize mode)...


Optimizing Reservoir: 100%|██████████| 120/120 [00:37<00:00,  3.19trial/s, best_score=0.00716]


Optimization complete! Best Parameters: {'spectral_radius': 0.6, 'leak_rate': 0.7000000000000001, 'input_scaling': 0.48}
Re-initializing current instance weights with the optimal configuration...


In [71]:
viz.compare_components([predictions, Y_val], labels=['predictions', 'true values'])

In [72]:
chaos_utils.lyapunov_time(predictions[:, 0], Y_val[:, 0])

73.0

In [78]:
viz.compare_plot([predictions, Y_train], legend_names=['predictions', 'true values'])